# Smart Plating

**Type:** Consumer — pulls from storage, stores output in dimensional depot  
**Blueprint:** `smart-plating` (6 Reinforced Iron Plates/min + 6 Rotors/min → 6 Smart Plating/min at 100%)  
**Pulls from:** Rotors storage (`parts/rotors.ipynb`), Reinforced Iron Plates storage (`parts/reinforced-iron-plates.ipynb`)

From 3 × 270 = 810 Iron Ingots/min (after all stash budgets) → **27.15 Smart Plating/min**

In [ ]:
from blueprints import BLUEPRINTS, Blueprint, Stage
from satisfactory import ITEMS

BP  = BLUEPRINTS
TOL = 0.5


def machine_hover(bp: Blueprint, output_key: str, target_rate: float) -> str:
    if not bp.stages:
        return ""
    rates = bp.stage_rates(output_key, target_rate)
    lines = ["── Machines (per copy) ──"]
    for r in rates:
        name = ITEMS[r["item_key"]].name
        lines.append(f"  {r['machines']}× {r['building'].title()} → {r['per_machine_rate']:.2f} {name}/min each")
    return "<br>" + "<br>".join(lines)

## Constants

Supply rates come from the upstream storage notebooks — copy the value printed at the bottom of each.

In [ ]:
# ── Supply from storage ───────────────────────────────────────────────────────
# parts/rotors.ipynb → available_for_smart_plating (rotor_total - 5 stash)
ROTORS_FROM_STORAGE = 27.16  # Rotors/min pulled from storage
# parts/reinforced-iron-plates.ipynb → available_for_smart_plating (reinforced_total - 5 stash)
RIPS_FROM_STORAGE   = 27.16  # Reinforced Iron Plates/min pulled from storage

# ── Blueprint placement ───────────────────────────────────────────────────────
SMART_PLATING_COPIES      = 5
SMART_PLATING_OUTPUT_EACH = 5.43  # Smart Plating/min — set this on each blueprint in-game

## Derived rates and balance

In [ ]:
smart_plating_total = SMART_PLATING_COPIES * SMART_PLATING_OUTPUT_EACH

rotors_consumed = smart_plating_total * BP['smart-plating'].input_ratio('rotor',                 'smart-plating')
rips_consumed   = smart_plating_total * BP['smart-plating'].input_ratio('reinforced-iron-plate', 'smart-plating')

assert abs(rotors_consumed - ROTORS_FROM_STORAGE) < TOL, (
    f"Rotor balance: consuming {rotors_consumed:.2f}/min, available {ROTORS_FROM_STORAGE:.2f}/min "
    f"(delta {rotors_consumed - ROTORS_FROM_STORAGE:+.2f})"
)
assert abs(rips_consumed - RIPS_FROM_STORAGE) < TOL, (
    f"Reinforced Iron Plate balance: consuming {rips_consumed:.2f}/min, available {RIPS_FROM_STORAGE:.2f}/min "
    f"(delta {rips_consumed - RIPS_FROM_STORAGE:+.2f})"
)

print(f"✓ Chain balanced")
print(f"  {ROTORS_FROM_STORAGE:.2f} Rotors/min  +  {RIPS_FROM_STORAGE:.2f} Reinforced Iron Plates/min")
print(f"  →  {smart_plating_total:.2f} Smart Plating/min")
print(f"  Set each of {SMART_PLATING_COPIES} blueprints to: {SMART_PLATING_OUTPUT_EACH:.2f} Smart Plating/min")

## Production flow

Arrow width is proportional to **ingot-equivalent cost** — Rotors cost 11.25 ingots each, Reinforced Iron Plates cost 12.00 each.  
Hover over any node or arrow for detail.

In [ ]:
import plotly.graph_objects as go

ingot_per_rod        = BP['iron-rod'].input_ratio('iron-ingot', 'iron-rod')
ingot_per_screw      = BP['screw'].input_ratio('iron-ingot', 'screw')
ingot_per_plate      = BP['iron-plate'].input_ratio('iron-ingot', 'iron-plate')
ingot_per_reinforced = (
    BP['reinforced-iron-plate'].input_ratio('iron-plate', 'reinforced-iron-plate') * ingot_per_plate +
    BP['reinforced-iron-plate'].input_ratio('screw',      'reinforced-iron-plate') * ingot_per_screw
)
ingot_per_rotor = (
    BP['rotor'].input_ratio('iron-rod', 'rotor') * ingot_per_rod +
    BP['rotor'].input_ratio('screw',    'rotor') * ingot_per_screw
)

STORAGE_COL = "#475569"
FINAL_COL   = "#b45309"
LINK_COL    = "rgba(148, 163, 184, 0.25)"

nodes = [
    (f"Rotors (storage)<br>{ROTORS_FROM_STORAGE:.2f}/min",
     STORAGE_COL,
     f"Pulling {ROTORS_FROM_STORAGE:.2f} Rotors/min from storage<br>"
     f"Ingot cost: {ingot_per_rotor:.2f} ingots/rotor  ({ROTORS_FROM_STORAGE * ingot_per_rotor:.2f} ingots/min)"),

    (f"Reinforced Iron Plates (storage)<br>{RIPS_FROM_STORAGE:.2f}/min",
     STORAGE_COL,
     f"Pulling {RIPS_FROM_STORAGE:.2f} Reinforced Iron Plates/min from storage<br>"
     f"Ingot cost: {ingot_per_reinforced:.2f} ingots/plate  ({RIPS_FROM_STORAGE * ingot_per_reinforced:.2f} ingots/min)"),

    (f"Smart Plating ×{SMART_PLATING_COPIES}<br>set each to {SMART_PLATING_OUTPUT_EACH:.2f}/min",
     FINAL_COL,
     f"{SMART_PLATING_COPIES} blueprints, each set to {SMART_PLATING_OUTPUT_EACH:.2f} Smart Plating/min<br>"
     f"Total: {smart_plating_total:.2f} Smart Plating/min"
     + machine_hover(BP['smart-plating'], 'smart-plating', SMART_PLATING_OUTPUT_EACH)),
]

links = [
    (0, 2, ROTORS_FROM_STORAGE * ingot_per_rotor,
     f"{ROTORS_FROM_STORAGE:.2f} Rotors/min → Smart Plating"),
    (1, 2, RIPS_FROM_STORAGE   * ingot_per_reinforced,
     f"{RIPS_FROM_STORAGE:.2f} Reinforced Iron Plates/min → Smart Plating"),
]

fig = go.Figure(go.Sankey(
    arrangement="snap",
    node=dict(
        pad=20, thickness=24,
        line=dict(color="#0f172a", width=1.5),
        label=[n[0] for n in nodes],
        color=[n[1] for n in nodes],
        customdata=[n[2] for n in nodes],
        hovertemplate="%{customdata}<extra></extra>",
    ),
    link=dict(
        source=[l[0] for l in links],
        target=[l[1] for l in links],
        value= [l[2] for l in links],
        customdata=[l[3] for l in links],
        color=LINK_COL,
        hovertemplate="%{customdata}<extra></extra>",
    ),
))

fig.update_layout(
    title=dict(
        text=f"Smart Plating — {smart_plating_total:.2f} Smart Plating/min from storage",
        font=dict(color="#e2e8f0", size=15), x=0.01,
    ),
    paper_bgcolor="#0f172a",
    font=dict(color="#e2e8f0", size=11, family="monospace"),
    height=400,
    margin=dict(l=20, r=20, t=50, b=20),
)

fig.show()